# **06. Gradio によるエージェント思考プロセスの可視化**

この回は，エージェントが内部でどのような「やり取り」をしているかを，Gradio を使ってブラウザ上で確認できる UI を構築します．
Executor の下書きと Critic の指摘、そして修正後の最終回答を一覧できるようにします．

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
print('永続ディレクトリ:', PERSIST_ROOT)
from src.common import load_llm, generate_text
from src.agent_core import LLMExecutorCriticAgent
from src.ui import create_agent_ui

model, tokenizer = load_llm()
print('準備完了')


In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 512, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

# エージェントを初期化
agent = LLMExecutorCriticAgent(llm_chat)
print('エージェントの準備が完了しました。')

## **1. エージェントUIの起動**
以下のセルを実行して，UIを立ち上げます．`full_log` を Markdown 形式で表示するように設定しています．

In [ ]:
def run_agent_for_ui(query):
    final_answer, full_log, steps = agent.run_pipeline(query)
    # ログを Markdown の引用形式に整形
    formatted_log = ""
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
    return final_answer, formatted_log

ui = create_agent_ui(run_agent_for_ui)
ui.launch(share=True, debug=True)

## **演習：エージェントの挙動をカスタマイズする**

`src/` フォルダのコードを書き換える必要はありません．以下のセルのように，新しい `RoleConfig` を作成してエージェントに渡すだけで，性格や厳格さを自由に変えることができます．

In [ ]:
# カスタムプロンプトの設定例
custom_roles = [
    RoleConfig(name="Executor", system_prompt="あなたは非常に丁寧な日本語で答える執事です。"),
    RoleConfig(name="Critic", system_prompt="あなたは間違いを絶対に見逃さない、非常に厳しい教育係です。不適切な表現や計算ミスがあれば、厳しく指摘してください。")
]

# 新しい性格のエージェントを作成
custom_agent = LLMExecutorCriticAgent(llm_chat, role_configs=custom_roles)

def run_custom_agent_ui(query):
    final_answer, full_log, steps = custom_agent.run_pipeline(query)
    formatted_log = ""
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
    return final_answer, formatted_log

ui_custom = create_agent_ui(run_custom_agent_ui)
ui_custom.launch(share=True, debug=True)

## **まとめ**
- UIを通じてエージェントの内部動作を可視化することで，どこで思考が脱線し，どこで修正されたかが明確になります．
- これは，実用的なAIシステムをデバッグ・改善する上で非常に重要なプロセスです．

最終回では，このエージェントに RAG（外部知識参照）を統合し，事実に基づいた回答と検証を行うシステムを完成させます．